# 3. Causal feature engineering

Five signals, each with a level and a 6-hour/24-hour deviation: 15 model features. Missing-history coverage is kept separately.

In [ ]:
from pathlib import Path
import json
import sys
import yaml

ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "pyproject.toml").exists()
)
sys.path.insert(0, str(ROOT / "src"))
EXPERIMENT = yaml.safe_load(
    (ROOT / "configs/synthetic_experiment.yml").read_text()
)
DATASET = ROOT / EXPERIMENT["dataset"]
RUN = ROOT / EXPERIMENT["output"]


In [ ]:
from telco_anomaly.synthetic_pipeline import (
    load_observations, build_features, feature_columns,
)
data, manifest = load_observations(DATASET)
features = build_features(
    data,
    cadence_seconds=manifest["config"]["sample_minutes"] * 60,
    minimum_coverage=EXPERIMENT["minimum_coverage"],
)
display(features.head())
display(features[feature_columns(features)].describe().T)
display(features.feature_coverage.describe())

Each deviation compares the current measurement with the median and IQR of **earlier** observations. Direction is downward for received power, upward for FEC, and two-sided for transmit power/current. FEC is an interval corrected-codeword count, transformed with `log1p(count / seconds)`; it is not differenced. No latent loss, sensitivity, fault labels, entity identifier or vendor name enters the model. Under 80% valid deviation features means abstention. Large gaps invalidate rolling history rather than forward-filling it.